In [ ]:
import xarray as xr
import matplotlib.pylab as plt
import xarray_regrid.methods.conservative  # pylint: disable=unused-import
from functools import partial
import xskillscore as xs
import pint_xarray
from dask.diagnostics import ProgressBar
import json
import pandas as pd
import numpy as np

from pism_terra.filtering import importance_sampling
from pism_terra.processing import preprocess_netcdf as preprocess

data_dir = "~/base/pism-terra"

pctls = [0.05, 0.95]
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

debm_uq_vars = {'surface.debm_simple.c1': 'c1', 
                'surface.debm_simple.c2': 'c2',
                'surface.debm_simple.air_temp_all_precip_as_snow': 'as_snow', 
                'surface.debm_simple.air_temp_all_precip_as_rain': 'as_rain', 
                'surface.debm_simple.refreeze': 'refreeze'}

pdd_uq_vars = {'surface.pdd.factor_ice': 'fice', 
               'surface.pdd.factor_snow': 'fsnow',
               'surface.pdd.refreeze': 'refreeze'}

m_vars = ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]

obs = xr.open_dataset(f"{data_dir}/2026_06_kitp_debm_calib/kitp/input/v4/HIRHAM5-ERA5_YMM_1990_2019_v4.nc", engine="netcdf4",
                     decode_times=False,
                     decode_timedelta=False, chunks=None).drop_dims("nv", errors="ignore")

obs = obs.pint.quantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[v] = obs[v].pint.to("kg m^-2 yr^-1")
obs = obs.pint.dequantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[f"{v}_error"] = xr.where(obs[v] != 0, 0.10 * obs[v], 1e-8)

for ebm, ebm_uq_vars, fudge_factor in zip(["debm"], [debm_uq_vars], [1000]):
    
    ds = xr.open_mfdataset(f"{data_dir}/2026_06_kitp_{ebm}_calib/output/processed_spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
                         preprocess=partial(preprocess,
                                            uq_regexp=None,
                                            exp_regexp="uq_(.+?)_"),
                        engine="netcdf4",
                        join="outer",
                        compat="no_conflicts",
                        parallel=True,
                        chunks="auto",
                        decode_times=False,
                        decode_timedelta=False).drop_dims("nv", errors="ignore")

    
    ebm_uq_df = ds.pism_config.to_series().apply(json.loads).apply(pd.Series)[ebm_uq_vars.keys()]
    ds["time"] = obs["time"]

    _obs = obs.regrid.conservative(ds.drop_vars("pism_config")).squeeze()
    mask = ds[m_vars].isel(exp_id=0).notnull()                                                                                                                                         
    _obs[m_vars] = _obs[m_vars].where(mask) 
    melt_mask = (_obs["climatic_mass_balance"].mean(dim="time") < 0)
    _obs[m_vars] = _obs[m_vars].where(melt_mask) 
    _ds = ds[m_vars].where(melt_mask)
    
    for v in m_vars:
    
        with ProgressBar():
            ebm_filtered = importance_sampling(_ds, _obs, 
                            sim_var=v,
                            obs_mean_var=v, 
                            obs_std_var=f"{v}_error",
                            sum_dims=['time', 'x', 'y'],
                            n_samples=ds.exp_id.size,
                            fudge_factor=fudge_factor
                                              )

            ebm_sampled_ids = ebm_filtered.exp_id_sampled.values
            ebm_counts = pd.Series(ebm_sampled_ids).value_counts()
                                                                                                                                                                                                 
            # Reindex config_df to the sampled IDs and plot histograms                                                                                                                         
            ds_sampled_configs = ebm_uq_df.loc[ebm_counts.index].reindex(ebm_counts.index)                                                                                                                
            most_sampled_id = ebm_counts.idxmax()                                                                                                                                  
            most_sampled_params = ebm_uq_df.loc[most_sampled_id]
            print(f"\n{ebm} / {v} — most sampled id={most_sampled_id} (count={ebm_counts.max()})")                                                                                 
            for k, short in ebm_uq_vars.items():                                                                                                                                   
                print(f"  {short}: {most_sampled_params[k]:.6g}")     
                  
            fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))           
            for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):                                                                                                                                             
                    # Repeat each parameter value by its sample count                                                                                                                              
                      values = np.repeat(ebm_uq_df[key].values,                                                                                                                                      
                                     ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int))                                                                                           
                      ax.hist(values, bins=15)                                                                                                                                                       
                      ax.set_xlabel(value)    
                      ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                      # ax.set_ylabel("Count")                                                                                                                                                         
            fig.tight_layout()   
            fig.savefig(f"{ebm}_{v}.png", dpi=300)
            plt.close()
            del fig
            
            
            # Plot smallest RMSE                                                                                                                                                       
            rmse = xs.rmse(_ds[v].mean(dim="time"), _obs[v].mean(dim="time"), dim=["x", "y"], skipna=True).compute()
            best_id = rmse.idxmin(dim="exp_id").values                                                                                                                                                                                                                                                                                                                          
            sim_best = _ds[v].sel(exp_id=best_id).mean(dim="time").squeeze()                                                                                                            
            obs_mean = _obs[v].mean(dim="time").squeeze()                                                                                                                               
            vmin = min(float(obs_mean.min()), float(sim_best.min()))
            vmax = max(float(obs_mean.max()), float(sim_best.max()))
            best_params = ebm_uq_df.loc[best_id]                                                                                                                                                                                         
            fig, axes = plt.subplots(1, 3, sharey=True, figsize=(12, 4))                                                                                                                         
            obs_mean.plot(ax=axes[0], vmin=vmin, vmax=vmax)                                                                                                                            
            axes[0].set_title("Observed")                                                                                                                                              
            sim_best.plot(ax=axes[1], vmin=vmin, vmax=vmax)                                                                                                                            
            param_str = ", ".join(f"{v}={best_params[k]:.4g}" for k, v in ebm_uq_vars.items())                                                                                                 
            axes[1].set_title(f"Best (id={best_id}, RMSE={float(rmse.sel(exp_id=best_id)):.1f})\n{param_str}") 
            (sim_best - obs_mean).plot(ax=axes[2], cmap="RdBu", vmin=-2000, vmax=2000)                                                                                                                        
            axes[2].set_title("Difference")                   
            fig.tight_layout()                                                                                                                                                         
            fig.savefig(f"{ebm}_{v}_best_rmse.png", dpi=300)                                                                                                                           
            plt.close()
            del fig 

In [ ]:
import xarray as xr
import matplotlib.pylab as plt
import xarray_regrid.methods.conservative  # pylint: disable=unused-import
from functools import partial
import xskillscore as xs
import pint_xarray
from dask.diagnostics import ProgressBar
import json
import pandas as pd
import numpy as np

from pism_terra.processing import preprocess_netcdf as preprocess


def decorrelation_length(field_2d, pixel_size, threshold=1.0 / np.e):
    """
    Radially-averaged spatial-ACF decorrelation length for a 2D field.

    Why: pixel-wise RMSE treats every cell as independent, but glaciological
    fields are smooth on scales of many cells. Returns the lag (in the units
    of ``pixel_size``) at which the autocorrelation first falls below
    ``threshold``; use that as the block side for bootstrap resampling.
    """
    a = np.asarray(field_2d, dtype=float)
    finite = np.isfinite(a)
    if not finite.any():
        return float("nan")
    a = np.where(finite, a, np.nanmean(a))
    a = a - a.mean()
    fft = np.fft.fft2(a)
    acf = np.fft.fftshift(np.fft.ifft2(fft * np.conj(fft)).real)
    acf = acf / acf.max()
    ny, nx = a.shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.indices(a.shape)
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2).astype(int)
    counts = np.maximum(np.bincount(r.ravel()), 1)
    radial = np.bincount(r.ravel(), weights=acf.ravel()) / counts
    rmax = min(cy, cx)
    radial = radial[: rmax + 1]
    below = np.where(radial < threshold)[0]
    lag_pixels = below[0] if below.size else rmax
    return float(lag_pixels) * float(pixel_size)


def block_bootstrap_rmse(sim, obs, block_size, n_boot=500, seed=0):
    """
    Block-bootstrap RMSE per experiment, returning an (exp_id, boot) array.

    ``sim`` is (exp_id, y, x), ``obs`` is (y, x). Blocks of side
    ``block_size`` pixels (≥ decorrelation length / pixel size) are resampled
    with replacement; one global RMSE is computed per resample per experiment.
    """
    sim_v = np.asarray(sim.values, dtype=float)
    obs_v = np.asarray(obs.values, dtype=float)
    sq_err = (sim_v - obs_v[None, :, :]) ** 2
    valid = np.isfinite(sq_err).all(axis=0)
    ny, nx = obs_v.shape
    by = max(1, ny // block_size)
    bx = max(1, nx // block_size)
    block_sums = np.zeros((sim_v.shape[0], by, bx))
    block_counts = np.zeros((by, bx), dtype=int)
    for i in range(by):
        for j in range(bx):
            ys = slice(i * block_size, (i + 1) * block_size)
            xs = slice(j * block_size, (j + 1) * block_size)
            v = valid[ys, xs]
            block_counts[i, j] = int(v.sum())
            chunk = np.where(v, sq_err[:, ys, xs], 0.0)
            block_sums[:, i, j] = chunk.sum(axis=(1, 2))
    block_sums = block_sums.reshape(sim_v.shape[0], -1)
    block_counts = block_counts.reshape(-1)
    valid_blocks = np.where(block_counts > 0)[0]
    rng = np.random.default_rng(seed)
    rmses = np.empty((sim_v.shape[0], n_boot))
    for b in range(n_boot):
        idx = rng.choice(valid_blocks, size=valid_blocks.size, replace=True)
        s = block_sums[:, idx].sum(axis=1)
        c = block_counts[idx].sum()
        rmses[:, b] = np.sqrt(s / max(c, 1))
    return xr.DataArray(
        rmses,
        dims=["exp_id", "boot"],
        coords={"exp_id": sim.exp_id, "boot": np.arange(n_boot)},
    )


data_dir = "~/base/pism-terra"

pctls = [0.05, 0.95]
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

debm_uq_vars = {
    "surface.debm_simple.c1": "c1",
    "surface.debm_simple.c2": "c2",
    "surface.debm_simple.air_temp_all_precip_as_snow": "as_snow",
    "surface.debm_simple.air_temp_all_precip_as_rain": "as_rain",
    "surface.debm_simple.refreeze": "refreeze",
}

pdd_uq_vars = {"surface.pdd.factor_ice": "fice", "surface.pdd.factor_snow": "fsnow", "surface.pdd.refreeze": "refreeze"}

m_vars = ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]

obs = xr.open_dataset(
    f"{data_dir}/2026_06_kitp_debm_calib/kitp/input/v4/HIRHAM5-ERA5_YMM_1990_2019_v4.nc",
    engine="netcdf4",
    decode_times=False,
    decode_timedelta=False,
    chunks=None,
).drop_dims("nv", errors="ignore")

obs = obs.pint.quantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[v] = obs[v].pint.to("kg m^-2 yr^-1")
obs = obs.pint.dequantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[f"{v}_error"] = xr.where(obs[v] != 0, 0.10 * obs[v], 1e-8)

for ebm, ebm_uq_vars, fudge_factor in zip(["debm"], [debm_uq_vars], [1000]):

    ds = xr.open_mfdataset(
        f"{data_dir}/2026_06_kitp_{ebm}_calib/output/processed_spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
        preprocess=partial(preprocess, uq_regexp=None, exp_regexp="uq_(.+?)_"),
        engine="netcdf4",
        join="outer",
        compat="no_conflicts",
        parallel=True,
        chunks="auto",
        decode_times=False,
        decode_timedelta=False,
    ).drop_dims("nv", errors="ignore")

    ebm_uq_df = ds.pism_config.to_series().apply(json.loads).apply(pd.Series)[ebm_uq_vars.keys()]
    ds["time"] = obs["time"]

    _obs = obs.regrid.conservative(ds.drop_vars("pism_config")).squeeze()
    mask = ds[m_vars].isel(exp_id=0).notnull()
    _obs[m_vars] = _obs[m_vars].where(mask)
    melt_mask = _obs["climatic_mass_balance"].mean(dim="time") < 0
    _obs[m_vars] = _obs[m_vars].where(melt_mask)
    _ds = ds[m_vars].where(melt_mask)

    for v in m_vars:

        with ProgressBar():

            fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))
            for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):
                # Repeat each parameter value by its sample count
                values = np.repeat(
                    ebm_uq_df[key].values, ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int)
                )
                ax.hist(values, bins=15)
                ax.set_xlabel(value)
                ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                # ax.set_ylabel("Count")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}.png", dpi=300)
            plt.close()
            del fig

            # 1) Decorrelation length from the observed time-mean field.
            sim_mean_all = _ds[v].mean(dim="time").compute()
            obs_mean = _obs[v].mean(dim="time").squeeze().compute()
            pixel_size = float(abs(_obs.x.diff("x").mean()))
            L = decorrelation_length(obs_mean.values, pixel_size)
            block_size = max(1, int(np.ceil(L / pixel_size)))
            print(f"{ebm}/{v}: decorrelation length ≈ {L:.0f} m, block_size = {block_size} px")

            # 2) Block-bootstrap RMSE per exp_id (honors spatial correlation).
            rmse_boot = block_bootstrap_rmse(sim_mean_all, obs_mean, block_size, n_boot=500)
            rmse_mean = rmse_boot.mean(dim="boot")
            rmse_lo = rmse_boot.quantile(0.05, dim="boot")
            rmse_hi = rmse_boot.quantile(0.95, dim="boot")

            # 3) Rank by bootstrap-mean RMSE; treat exp_ids whose CI overlaps
            # the leader's upper bound as statistically tied with the best.
            best_id = rmse_mean.idxmin(dim="exp_id").values
            best_hi = float(rmse_hi.sel(exp_id=best_id))
            tied_mask = rmse_lo <= best_hi
            tied_ids = list(rmse_mean.exp_id.where(tied_mask, drop=True).values)
            print(f"{ebm}/{v}: best exp_id = {best_id}, n tied within 5-95% CI = {len(tied_ids)}")

            # Write per-experiment stats to CSV so the user can inspect ties.
            rmse_df = pd.DataFrame(
                {
                    "rmse_mean": rmse_mean.values,
                    "rmse_lo": rmse_lo.values,
                    "rmse_hi": rmse_hi.values,
                    "tied_with_best": tied_mask.values,
                },
                index=pd.Index(rmse_mean.exp_id.values, name="exp_id"),
            ).join(ebm_uq_df, how="left").sort_values("rmse_mean")
            rmse_df.to_csv(f"{ebm}_{v}_rmse.csv")

            sim_best = _ds[v].sel(exp_id=best_id).mean(dim="time").squeeze()
            vmin = min(float(obs_mean.min()), float(sim_best.min()))
            vmax = max(float(obs_mean.max()), float(sim_best.max()))
            best_params = ebm_uq_df.loc[best_id]
            fig, axes = plt.subplots(1, 3, sharey=True, figsize=(12, 4))
            obs_mean.plot(ax=axes[0], vmin=vmin, vmax=vmax)
            axes[0].set_title("Observed")
            sim_best.plot(ax=axes[1], vmin=vmin, vmax=vmax)
            param_str = ", ".join(f"{v}={best_params[k]:.4g}" for k, v in ebm_uq_vars.items())
            rmse_best_mean = float(rmse_mean.sel(exp_id=best_id))
            rmse_best_lo = float(rmse_lo.sel(exp_id=best_id))
            rmse_best_hi = float(rmse_hi.sel(exp_id=best_id))
            axes[1].set_title(
                f"Best (id={best_id}, RMSE={rmse_best_mean:.1f} "
                f"[{rmse_best_lo:.1f}-{rmse_best_hi:.1f}], n_tied={len(tied_ids)})\n{param_str}"
            )
            (sim_best - obs_mean).plot(ax=axes[2], cmap="RdBu", vmin=-2000, vmax=2000)
            axes[2].set_title("Difference")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}_best_rmse.png", dpi=300)
            plt.close()
            del fig


In [ ]:
_obs["climatic_mass_balance"].mean(dim="time").plot()

In [ ]:
sampled_ids = debm_filtered.exp_id_sampled.values
counts = pd.Series(sampled_ids).value_counts()
                                                                                                                                                                                     
# Reindex config_df to the sampled IDs and plot histograms                                                                                                                         
sampled_configs = debm_uq_df.loc[counts.index].reindex(counts.index)                                                                                                                
                                                                                                                                                                                     
fig, axes = plt.subplots(2, 3, figsize=(12, 4))           
for ax, var in zip(axes.flat, debm_uq_vars):                                                                                                                                             
        # Repeat each parameter value by its sample count                                                                                                                              
          values = np.repeat(debm_uq_df[var].values,                                                                                                                                      
                         counts.reindex(debm_uq_df.index, fill_value=0).values.astype(int))                                                                                           
          ax.hist(values, bins=15)                                                                                                                                                       
          ax.set_xlabel(var)                                    
          ax.set_ylabel("Count")                                                                                                                                                         
plt.tight_layout()          

In [ ]:
_debm_filtered.exp_id_sampled

In [ ]:
df = pdd.pism_config.to_series().apply(json.loads).apply(pd.Series)[pdd_uq_vars]

In [ ]:
import pandas as pd

In [ ]:
            rmse = xs.rmse(_ds[v].mean(dim="time"), _obs[v].mean(dim="time"), dim=["x", "y"], skipna=True).compute()
            best_id = rmse.idxmin(dim="exp_id").values   

In [ ]:
best_id

In [ ]:
ds = xr.open_dataset("/Volumes/LHS800DATA/2026_04_kitp_debm/output/spatial/fldsum_spatial_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc")
ds_scalar = xr.open_dataset("/Volumes/LHS800DATA/2026_04_kitp_debm/output/scalar/scalar_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc")
ds_scalar = ds_scalar.resample(time="YE").mean()

ds_new = xr.open_dataset("/Volumes/LHS800DATA/2026_04_kitp_debm_strong/output/spatial/fldsum_spatial_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0301-01-01.nc")
ds_new = ds_new.sel(basin="GIS")
#ds_new = ds_new.resample(time="YE").mean()


fig, ax = plt.subplots(1, 1)
#ds.sel(basin="GIS").tendency_of_ice_mass.plot(ax=ax)
#ds_scalar.tendency_of_ice_mass.plot(ax=ax)
ds_new.tendency_of_ice_mass.plot(ax=ax)



In [ ]:
ds_scalar.tendency_of_ice_mass.plot(ax=ax)

In [ ]:
ds_scalar.tendency_of_ice_mass.plot()

In [ ]:
ds_sampled_configs

In [ ]:
ds = xr.open_dataset("/Volumes/LHS800DATA/2026_04_kitp_debm_mode/output/spatial/fldsum_spatial_g1200m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0005-01-01.nc")
ds.sel(basin="GIS").tendency_of_ice_mass.plot()

In [ ]:
sds = xr.open_dataset("../fldsum_spatial_g2400m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc")

In [ ]:
sds.tendency_of_ice_mass.mean(dim="time")

In [ ]:
import xarray as xr
import matplotlib.pylab as plt
import xarray_regrid.methods.conservative  # pylint: disable=unused-import
from functools import partial
import xskillscore as xs
import pint_xarray
from dask.diagnostics import ProgressBar
import json
import pandas as pd
import numpy as np

from pism_terra.processing import preprocess_netcdf as preprocess


def decorrelation_length(field_2d, pixel_size, threshold=1.0 / np.e):
    """
    Radially-averaged spatial-ACF decorrelation length for a 2D field.

    Why: pixel-wise RMSE treats every cell as independent, but glaciological
    fields are smooth on scales of many cells. Returns the lag (in the units
    of ``pixel_size``) at which the autocorrelation first falls below
    ``threshold``; use that as the block side for bootstrap resampling.
    """
    a = np.asarray(field_2d, dtype=float)
    finite = np.isfinite(a)
    if not finite.any():
        return float("nan")
    a = np.where(finite, a, np.nanmean(a))
    a = a - a.mean()
    fft = np.fft.fft2(a)
    acf = np.fft.fftshift(np.fft.ifft2(fft * np.conj(fft)).real)
    acf = acf / acf.max()
    ny, nx = a.shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.indices(a.shape)
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2).astype(int)
    counts = np.maximum(np.bincount(r.ravel()), 1)
    radial = np.bincount(r.ravel(), weights=acf.ravel()) / counts
    rmax = min(cy, cx)
    radial = radial[: rmax + 1]
    below = np.where(radial < threshold)[0]
    lag_pixels = below[0] if below.size else rmax
    return float(lag_pixels) * float(pixel_size)


def block_bootstrap_rmse(sim, obs, block_size, n_boot=500, seed=0):
    """
    Block-bootstrap RMSE per experiment, returning an (exp_id, boot) array.

    ``sim`` is (exp_id, y, x), ``obs`` is (y, x). Blocks of side
    ``block_size`` pixels (≥ decorrelation length / pixel size) are resampled
    with replacement; one global RMSE is computed per resample per experiment.
    """
    sim_v = np.asarray(sim.values, dtype=float)
    obs_v = np.asarray(obs.values, dtype=float)
    sq_err = (sim_v - obs_v[None, :, :]) ** 2
    valid = np.isfinite(sq_err).all(axis=0)
    ny, nx = obs_v.shape
    by = max(1, ny // block_size)
    bx = max(1, nx // block_size)
    block_sums = np.zeros((sim_v.shape[0], by, bx))
    block_counts = np.zeros((by, bx), dtype=int)
    for i in range(by):
        for j in range(bx):
            ys = slice(i * block_size, (i + 1) * block_size)
            xs = slice(j * block_size, (j + 1) * block_size)
            v = valid[ys, xs]
            block_counts[i, j] = int(v.sum())
            chunk = np.where(v, sq_err[:, ys, xs], 0.0)
            block_sums[:, i, j] = chunk.sum(axis=(1, 2))
    block_sums = block_sums.reshape(sim_v.shape[0], -1)
    block_counts = block_counts.reshape(-1)
    valid_blocks = np.where(block_counts > 0)[0]
    rng = np.random.default_rng(seed)
    rmses = np.empty((sim_v.shape[0], n_boot))
    for b in range(n_boot):
        idx = rng.choice(valid_blocks, size=valid_blocks.size, replace=True)
        s = block_sums[:, idx].sum(axis=1)
        c = block_counts[idx].sum()
        rmses[:, b] = np.sqrt(s / max(c, 1))
    return xr.DataArray(
        rmses,
        dims=["exp_id", "boot"],
        coords={"exp_id": sim.exp_id, "boot": np.arange(n_boot)},
    )


data_dir = "~/base/pism-terra"

pctls = [0.05, 0.95]
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

debm_uq_vars = {
    "surface.debm_simple.c1": "c1",
    "surface.debm_simple.c2": "c2",
    "surface.debm_simple.air_temp_all_precip_as_snow": "as_snow",
    "surface.debm_simple.air_temp_all_precip_as_rain": "as_rain",
    "surface.debm_simple.refreeze": "refreeze",
}

pdd_uq_vars = {"surface.pdd.factor_ice": "fice", "surface.pdd.factor_snow": "fsnow", "surface.pdd.refreeze": "refreeze"}

m_vars = ["surface_melt_flux", "surface_runoff_flux","climatic_mass_balance"]

obs = xr.open_dataset(
    f"{data_dir}/2026_06_kitp_debm_calib/kitp/input/v4/HIRHAM5-ERA5_YMM_1990_2019_v4.nc",
    engine="netcdf4",
    decode_times=False,
    decode_timedelta=False,
    chunks=None,
).drop_dims("nv", errors="ignore")

obs = obs.pint.quantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[v] = obs[v].pint.to("kg m^-2 yr^-1")
obs = obs.pint.dequantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[f"{v}_error"] = xr.where(obs[v] != 0, 0.10 * obs[v], 1e-8)

for ebm, ebm_uq_vars, fudge_factor in zip(["debm"], [debm_uq_vars], [1000]):

    ds = xr.open_mfdataset(
        f"{data_dir}/2026_06_kitp_{ebm}_calib/output/processed_spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
        preprocess=partial(preprocess, uq_regexp=None, exp_regexp="uq_(.+?)_"),
        engine="netcdf4",
        join="outer",
        compat="no_conflicts",
        parallel=True,
        chunks="auto",
        decode_times=False,
        decode_timedelta=False,
    ).drop_dims("nv", errors="ignore")

    ebm_uq_df = ds.pism_config.to_series().apply(json.loads).apply(pd.Series)[ebm_uq_vars.keys()]
    ds["time"] = obs["time"]

    _obs = obs.regrid.conservative(ds.drop_vars("pism_config")).squeeze()
    mask = ds[m_vars].isel(exp_id=0).notnull()
    _obs[m_vars] = _obs[m_vars].where(mask)
    melt_mask = _obs["climatic_mass_balance"].mean(dim="time") < 1e36
    _obs[m_vars] = _obs[m_vars].where(melt_mask)
    _ds = ds[m_vars].where(melt_mask)

    for v in m_vars:

        with ProgressBar():

            # 1) Decorrelation length from the observed time-mean field.
            sim_mean_all = _ds[v].mean(dim="time").compute()
            obs_mean = _obs[v].mean(dim="time").squeeze().compute()
            pixel_size = float(abs(_obs.x.diff("x").mean()))
            L = decorrelation_length(obs_mean.values, pixel_size)
            block_size = max(1, int(np.ceil(L / pixel_size)))
            print(f"{ebm}/{v}: decorrelation length ≈ {L:.0f} m, block_size = {block_size} px")

            # 2) Block-bootstrap RMSE per exp_id (honors spatial correlation).
            rmse_boot = block_bootstrap_rmse(sim_mean_all, obs_mean, block_size, n_boot=500)
            rmse_mean = rmse_boot.mean(dim="boot")
            rmse_lo = rmse_boot.quantile(0.05, dim="boot")
            rmse_hi = rmse_boot.quantile(0.95, dim="boot")

            # 3) Rank by bootstrap-mean RMSE; treat exp_ids whose CI overlaps
            # the leader's upper bound as statistically tied with the best.
            best_id = rmse_mean.idxmin(dim="exp_id").values
            best_hi = float(rmse_hi.sel(exp_id=best_id))
            tied_mask = rmse_lo <= best_hi
            tied_ids = list(rmse_mean.exp_id.where(tied_mask, drop=True).values)
            print(f"{ebm}/{v}: best exp_id = {best_id}, n tied within 5-95% CI = {len(tied_ids)}")

            # Per-experiment weight for the parameter histograms: 1 if the
            # exp_id is in the statistically-tied set, 0 otherwise. This is
            # what ``np.repeat`` consumes below so each parameter value
            # contributes to the histogram only if its experiment passed the
            # bootstrap tie test.
            ebm_counts = pd.Series(
                tied_mask.values.astype(int),
                index=pd.Index(rmse_mean.exp_id.values, name="exp_id"),
            )

            fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))
            for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):
                # Repeat each parameter value by its sample count (= 1 if
                # the experiment tied with the best, 0 otherwise).
                values = np.repeat(
                    ebm_uq_df[key].values, ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int)
                )
                ax.hist(values, bins=15)
                ax.set_xlabel(value)
                ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                # ax.set_ylabel("Count")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}.png", dpi=300)
            plt.close()
            del fig

            # Write per-experiment stats to CSV so the user can inspect ties.
            rmse_df = pd.DataFrame(
                {
                    "rmse_mean": rmse_mean.values,
                    "rmse_lo": rmse_lo.values,
                    "rmse_hi": rmse_hi.values,
                    "tied_with_best": tied_mask.values,
                },
                index=pd.Index(rmse_mean.exp_id.values, name="exp_id"),
            ).join(ebm_uq_df, how="left").sort_values("rmse_mean")
            rmse_df.to_csv(f"{ebm}_{v}_rmse.csv")

            sim_best = _ds[v].sel(exp_id=best_id).mean(dim="time").squeeze()
            vmin = min(float(obs_mean.min()), float(sim_best.min()))
            vmax = max(float(obs_mean.max()), float(sim_best.max()))
            best_params = ebm_uq_df.loc[best_id]
            fig, axes = plt.subplots(1, 3, sharey=True, figsize=(12, 4))
            obs_mean.plot(ax=axes[0], vmin=vmin, vmax=vmax)
            axes[0].set_title("Observed")
            sim_best.plot(ax=axes[1], vmin=vmin, vmax=vmax)
            param_str = ", ".join(f"{v}={best_params[k]:.4g}" for k, v in ebm_uq_vars.items())
            rmse_best_mean = float(rmse_mean.sel(exp_id=best_id))
            rmse_best_lo = float(rmse_lo.sel(exp_id=best_id))
            rmse_best_hi = float(rmse_hi.sel(exp_id=best_id))
            axes[1].set_title(
                f"Best (id={best_id}, RMSE={rmse_best_mean:.1f} "
                f"[{rmse_best_lo:.1f}-{rmse_best_hi:.1f}], n_tied={len(tied_ids)})\n{param_str}"
            )
            (sim_best - obs_mean).plot(ax=axes[2], cmap="RdBu", vmin=-500, vmax=500)
            axes[2].set_title("Difference (sim-obs)")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}_best_rmse.png", dpi=300)
            plt.close()
            del fig


In [ ]:
_obs.time

In [ ]:
m = obs.climatic_mass_balance.mean(dim="time")

In [ ]:
m.where(m!=0).plot(vmin=-500,vmax=500)

In [ ]:
obs_mean.sum()

In [ ]:
_ds

In [ ]:
ds.data_vars

In [ ]:
obs

In [ ]:
_obs.climatic_mass_balance_error.mean(dim="time").plot()

In [ ]:
 area = ( 4800**2 * mask.mean(dim="time").sum(dim=["x", "y"]).climatic_mass_balance.compute().pint.dequantify()).pint.quantify("m^2")

In [ ]:
getattr(mask.pism_config.config, "grid.dx")

In [ ]:
(_obs.climatic_mass_balance.mean(dim="time").sum(dim=["x", "y"]).pint.quantify("kg/m^2/yr") / area).pint.to("Gt/yr").compute()

In [ ]:
(_obs.climatic_mass_balance.mean(dim="time").sum(dim=["x", "y"]).compute().pint.quantify() * area).pint.to("Gt/yr") / 365

In [ ]:
ds.exp_id.astype("int")

In [ ]:
import xarray as xr
import matplotlib.pylab as plt
import xarray_regrid.methods.conservative  # pylint: disable=unused-import
from functools import partial
import xskillscore as xs
import pint_xarray
from dask.diagnostics import ProgressBar
import json
import pandas as pd
import numpy as np

from pism_terra.processing import preprocess_netcdf as preprocess


def decorrelation_length(field_2d, pixel_size, threshold=1.0 / np.e):
    """
    Radially-averaged spatial-ACF decorrelation length for a 2D field.

    Why: pixel-wise RMSE treats every cell as independent, but glaciological
    fields are smooth on scales of many cells. Returns the lag (in the units
    of ``pixel_size``) at which the autocorrelation first falls below
    ``threshold``; use that as the block side for bootstrap resampling.
    """
    a = np.asarray(field_2d, dtype=float)
    finite = np.isfinite(a)
    if not finite.any():
        return float("nan")
    a = np.where(finite, a, np.nanmean(a))
    a = a - a.mean()
    fft = np.fft.fft2(a)
    acf = np.fft.fftshift(np.fft.ifft2(fft * np.conj(fft)).real)
    acf = acf / acf.max()
    ny, nx = a.shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.indices(a.shape)
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2).astype(int)
    counts = np.maximum(np.bincount(r.ravel()), 1)
    radial = np.bincount(r.ravel(), weights=acf.ravel()) / counts
    rmax = min(cy, cx)
    radial = radial[: rmax + 1]
    below = np.where(radial < threshold)[0]
    lag_pixels = below[0] if below.size else rmax
    return float(lag_pixels) * float(pixel_size)


def block_bootstrap_rmse(sim, obs, block_size, n_boot=500, seed=0):
    """
    Block-bootstrap RMSE per experiment, returning an (exp_id, boot) array.

    ``sim`` is (exp_id, y, x), ``obs`` is (y, x). Blocks of side
    ``block_size`` pixels (≥ decorrelation length / pixel size) are resampled
    with replacement; one global RMSE is computed per resample per experiment.
    """
    sim_v = np.asarray(sim.values, dtype=float)
    obs_v = np.asarray(obs.values, dtype=float)
    sq_err = (sim_v - obs_v[None, :, :]) ** 2
    valid = np.isfinite(sq_err).all(axis=0)
    ny, nx = obs_v.shape
    by = max(1, ny // block_size)
    bx = max(1, nx // block_size)
    block_sums = np.zeros((sim_v.shape[0], by, bx))
    block_counts = np.zeros((by, bx), dtype=int)
    for i in range(by):
        for j in range(bx):
            ys = slice(i * block_size, (i + 1) * block_size)
            xs = slice(j * block_size, (j + 1) * block_size)
            v = valid[ys, xs]
            block_counts[i, j] = int(v.sum())
            chunk = np.where(v, sq_err[:, ys, xs], 0.0)
            block_sums[:, i, j] = chunk.sum(axis=(1, 2))
    block_sums = block_sums.reshape(sim_v.shape[0], -1)
    block_counts = block_counts.reshape(-1)
    valid_blocks = np.where(block_counts > 0)[0]
    rng = np.random.default_rng(seed)
    rmses = np.empty((sim_v.shape[0], n_boot))
    for b in range(n_boot):
        idx = rng.choice(valid_blocks, size=valid_blocks.size, replace=True)
        s = block_sums[:, idx].sum(axis=1)
        c = block_counts[idx].sum()
        rmses[:, b] = np.sqrt(s / max(c, 1))
    return xr.DataArray(
        rmses,
        dims=["exp_id", "boot"],
        coords={"exp_id": sim.exp_id, "boot": np.arange(n_boot)},
    )


data_dir = "~/base/pism-terra"

pctls = [0.05, 0.95]
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

debm_uq_vars = {
    "surface.debm_simple.c1": "c1",
    "surface.debm_simple.c2": "c2",
    "surface.debm_simple.air_temp_all_precip_as_snow": "as_snow",
    "surface.debm_simple.air_temp_all_precip_as_rain": "as_rain",
    "surface.debm_simple.refreeze": "refreeze",
}

pdd_uq_vars = {"surface.pdd.factor_ice": "fice", "surface.pdd.factor_snow": "fsnow", "surface.pdd.refreeze": "refreeze"}

m_vars = ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]

obs = xr.open_dataset(
    f"{data_dir}/2026_06_kitp_debm_calib/kitp/input/v4/HIRHAM5-ERA5_YMM_1990_2019_v4.nc",
    engine="netcdf4",
    decode_times=False,
    decode_timedelta=False,
    chunks=None,
).drop_dims("nv", errors="ignore")

obs = obs.pint.quantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[v] = obs[v].pint.to("kg m^-2 yr^-1")
obs = obs.pint.dequantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[f"{v}_error"] = xr.where(obs[v] != 0, 0.10 * obs[v], 1e-8)

for ebm, ebm_uq_vars, fudge_factor in zip(["debm"], [debm_uq_vars], [10]):

    ds = (
        xr.open_mfdataset(
            f"{data_dir}/2026_06_kitp_{ebm}_calib/output/processed_spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
            preprocess=partial(preprocess, uq_regexp=None, exp_regexp="uq_(.+?)_"),
            engine="netcdf4",
            join="outer",
            compat="no_conflicts",
            parallel=True,
            chunks="auto",
            decode_times=False,
            decode_timedelta=False,
        )
        .drop_dims("nv", errors="ignore")
        .pint.quantify()
    )
    for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
        ds[v] = obs[v].pint.to("kg m^-2 yr^-1")

    ebm_uq_df = ds.pism_config.to_series().apply(json.loads).apply(pd.Series)[ebm_uq_vars.keys()]
    ds["time"] = obs["time"]

    _obs = obs.regrid.conservative(ds.drop_vars("pism_config")).squeeze()
    mask = ds[m_vars].isel(exp_id=0).notnull()
    _obs[m_vars] = _obs[m_vars].where(mask)
    melt_mask = _obs["climatic_mass_balance"].mean(dim="time") < 1e36
    _obs[m_vars] = _obs[m_vars].where(melt_mask)
    _ds = ds[m_vars].where(melt_mask)

    for v in m_vars:

        with ProgressBar():

            # 1) Decorrelation length from the observed time-mean field.
            sim_mean_all = _ds[v].mean(dim="time").compute()
            obs_mean = _obs[v].mean(dim="time").squeeze().compute()
            pixel_size = float(abs(_obs.x.diff("x").mean()))
            L = decorrelation_length(obs_mean.values, pixel_size)
            block_size = max(1, int(np.ceil(L / pixel_size)))
            print(f"{ebm}/{v}: decorrelation length ≈ {L:.0f} m, block_size = {block_size} px")

            # 2) Block-bootstrap RMSE per exp_id (honors spatial correlation).
            rmse_boot = block_bootstrap_rmse(sim_mean_all, obs_mean, block_size, n_boot=500)
            rmse_mean = rmse_boot.mean(dim="boot")
            rmse_lo = rmse_boot.quantile(0.05, dim="boot")
            rmse_hi = rmse_boot.quantile(0.95, dim="boot")

            # 3) Rank by bootstrap-mean RMSE; treat exp_ids whose CI overlaps
            # the leader's upper bound as statistically tied with the best.
            best_id = rmse_mean.idxmin(dim="exp_id").values
            best_hi = float(rmse_hi.sel(exp_id=best_id))
            tied_mask = rmse_lo <= best_hi
            tied_ids = list(rmse_mean.exp_id.where(tied_mask, drop=True).values)
            print(f"{ebm}/{v}: best exp_id = {best_id}, n tied within 5-95% CI = {len(tied_ids)}")

            # Per-experiment weight for the parameter histograms: 1 if the
            # exp_id is in the statistically-tied set, 0 otherwise. This is
            # what ``np.repeat`` consumes below so each parameter value
            # contributes to the histogram only if its experiment passed the
            # bootstrap tie test.
            ebm_counts = pd.Series(
                tied_mask.values.astype(int),
                index=pd.Index(rmse_mean.exp_id.values, name="exp_id"),
            )

            fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))
            for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):
                # Repeat each parameter value by its sample count (= 1 if
                # the experiment tied with the best, 0 otherwise).
                values = np.repeat(
                    ebm_uq_df[key].values, ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int)
                )
                ax.hist(values, bins=15)
                ax.set_xlabel(value)
                ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                # ax.set_ylabel("Count")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}.png", dpi=300)
            plt.close()
            del fig

            # Write per-experiment stats to CSV so the user can inspect ties.
            rmse_df = (
                pd.DataFrame(
                    {
                        "rmse_mean": rmse_mean.values,
                        "rmse_lo": rmse_lo.values,
                        "rmse_hi": rmse_hi.values,
                        "tied_with_best": tied_mask.values,
                    },
                    index=pd.Index(rmse_mean.exp_id.values, name="exp_id"),
                )
                .join(ebm_uq_df, how="left")
                .sort_values("rmse_mean")
            )
            rmse_df.to_csv(f"{ebm}_{v}_rmse.csv")

            sim_best = _ds[v].sel(exp_id=best_id).mean(dim="time").squeeze()
            vmin = min(float(obs_mean.min()), float(sim_best.min()))
            vmax = max(float(obs_mean.max()), float(sim_best.max()))
            best_params = ebm_uq_df.loc[best_id]
            fig, axes = plt.subplots(1, 3, sharey=True, figsize=(12, 4))
            obs_mean.plot(ax=axes[0], vmin=vmin, vmax=vmax)
            axes[0].set_title("Observed")
            sim_best.plot(ax=axes[1], vmin=vmin, vmax=vmax)
            param_str = ", ".join(f"{v}={best_params[k]:.4g}" for k, v in ebm_uq_vars.items())
            rmse_best_mean = float(rmse_mean.sel(exp_id=best_id))
            rmse_best_lo = float(rmse_lo.sel(exp_id=best_id))
            rmse_best_hi = float(rmse_hi.sel(exp_id=best_id))
            axes[1].set_title(
                f"Best (id={best_id}, RMSE={rmse_best_mean:.1f} "
                f"[{rmse_best_lo:.1f}-{rmse_best_hi:.1f}], n_tied={len(tied_ids)})\n{param_str}"
            )
            (sim_best - obs_mean).plot(ax=axes[2], cmap="RdBu", vmin=-1000, vmax=1000)
            axes[2].set_title("Difference")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}_best_rmse.png", dpi=300)
            plt.close()
            del fig


In [ ]:
ds.surface_runoff_flux

In [ ]:
import xarray as xr
import matplotlib.pylab as plt
import xarray_regrid.methods.conservative  # pylint: disable=unused-import
from functools import partial
import xskillscore as xs
import pint_xarray
from dask.diagnostics import ProgressBar
import json
import pandas as pd
import numpy as np

from pism_terra.processing import preprocess_netcdf as preprocess


def decorrelation_length(field_2d, pixel_size, threshold=1.0 / np.e):
    """
    Radially-averaged spatial-ACF decorrelation length for a 2D field.

    Why: pixel-wise RMSE treats every cell as independent, but glaciological
    fields are smooth on scales of many cells. Returns the lag (in the units
    of ``pixel_size``) at which the autocorrelation first falls below
    ``threshold``; use that as the block side for bootstrap resampling.
    """
    a = np.asarray(field_2d, dtype=float)
    finite = np.isfinite(a)
    if not finite.any():
        return float("nan")
    a = np.where(finite, a, np.nanmean(a))
    a = a - a.mean()
    fft = np.fft.fft2(a)
    acf = np.fft.fftshift(np.fft.ifft2(fft * np.conj(fft)).real)
    acf = acf / acf.max()
    ny, nx = a.shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.indices(a.shape)
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2).astype(int)
    counts = np.maximum(np.bincount(r.ravel()), 1)
    radial = np.bincount(r.ravel(), weights=acf.ravel()) / counts
    rmax = min(cy, cx)
    radial = radial[: rmax + 1]
    below = np.where(radial < threshold)[0]
    lag_pixels = below[0] if below.size else rmax
    return float(lag_pixels) * float(pixel_size)


def block_bootstrap_rmse(sim, obs, block_size, n_boot=500, seed=0):
    """
    Block-bootstrap RMSE per experiment, returning an (exp_id, boot) array.

    ``sim`` is (exp_id, y, x), ``obs`` is (y, x). Blocks of side
    ``block_size`` pixels (≥ decorrelation length / pixel size) are resampled
    with replacement; one global RMSE is computed per resample per experiment.
    """
    sim_v = np.asarray(sim.values, dtype=float)
    obs_v = np.asarray(obs.values, dtype=float)
    sq_err = (sim_v - obs_v[None, :, :]) ** 2
    valid = np.isfinite(sq_err).all(axis=0)
    ny, nx = obs_v.shape
    by = max(1, ny // block_size)
    bx = max(1, nx // block_size)
    block_sums = np.zeros((sim_v.shape[0], by, bx))
    block_counts = np.zeros((by, bx), dtype=int)
    for i in range(by):
        for j in range(bx):
            ys = slice(i * block_size, (i + 1) * block_size)
            xs = slice(j * block_size, (j + 1) * block_size)
            v = valid[ys, xs]
            block_counts[i, j] = int(v.sum())
            chunk = np.where(v, sq_err[:, ys, xs], 0.0)
            block_sums[:, i, j] = chunk.sum(axis=(1, 2))
    block_sums = block_sums.reshape(sim_v.shape[0], -1)
    block_counts = block_counts.reshape(-1)
    valid_blocks = np.where(block_counts > 0)[0]
    rng = np.random.default_rng(seed)
    rmses = np.empty((sim_v.shape[0], n_boot))
    for b in range(n_boot):
        idx = rng.choice(valid_blocks, size=valid_blocks.size, replace=True)
        s = block_sums[:, idx].sum(axis=1)
        c = block_counts[idx].sum()
        rmses[:, b] = np.sqrt(s / max(c, 1))
    return xr.DataArray(
        rmses,
        dims=["exp_id", "boot"],
        coords={"exp_id": sim.exp_id, "boot": np.arange(n_boot)},
    )


data_dir = "~/base/pism-terra"

pctls = [0.05, 0.95]
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

debm_uq_vars = {
    "surface.debm_simple.c1": "c1",
    "surface.debm_simple.c2": "c2",
    "surface.debm_simple.air_temp_all_precip_as_snow": "as_snow",
    "surface.debm_simple.air_temp_all_precip_as_rain": "as_rain",
    "surface.debm_simple.refreeze": "refreeze",
}

pdd_uq_vars = {"surface.pdd.factor_ice": "fice", "surface.pdd.factor_snow": "fsnow", "surface.pdd.refreeze": "refreeze"}

m_vars = ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]

obs = xr.open_dataset(
    f"{data_dir}/2026_06_kitp_debm_calib/kitp/input/v4/HIRHAM5-ERA5_YMM_1990_2019_v4.nc",
    engine="netcdf4",
    decode_times=False,
    decode_timedelta=False,
    chunks=None,
).drop_dims("nv", errors="ignore")

obs = obs.pint.quantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[v] = obs[v].pint.to("kg m^-2 yr^-1")
obs = obs.pint.dequantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[f"{v}_error"] = xr.where(obs[v] != 0, 0.10 * obs[v], 1e-8)

for ebm, ebm_uq_vars, fudge_factor in zip(["debm"], [debm_uq_vars], [10]):

    ds = (
        xr.open_mfdataset(
            f"{data_dir}/2026_06_kitp_{ebm}_calib/output/processed_spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
            preprocess=partial(preprocess, uq_regexp=None, exp_regexp="uq_(.+?)_"),
            engine="netcdf4",
            join="outer",
            compat="no_conflicts",
            parallel=True,
            chunks="auto",
            decode_times=False,
            decode_timedelta=False,
        )
        .drop_dims("nv", errors="ignore")
        .pint.quantify()
    )
    for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
        ds[v] = ds[v].pint.to("kg m^-2 yr^-1")
    ds = ds.pint.dequantify()

    ebm_uq_df = ds.pism_config.to_series().apply(json.loads).apply(pd.Series)[ebm_uq_vars.keys()]
    ds["time"] = obs["time"]

    _obs = obs.regrid.conservative(ds.drop_vars("pism_config")).squeeze()
    mask = ds[m_vars].isel(exp_id=0).notnull()
    _obs[m_vars] = _obs[m_vars].where(mask)
    melt_mask = _obs["climatic_mass_balance"].mean(dim="time") < 1e36
    _obs[m_vars] = _obs[m_vars].where(melt_mask)
    _ds = ds[m_vars].where(melt_mask)

    for v in m_vars:

        with ProgressBar():

            # 1) Decorrelation length from the observed time-mean field.
            sim_mean_all = _ds[v].mean(dim="time").compute()
            obs_mean = _obs[v].mean(dim="time").squeeze().compute()
            pixel_size = float(abs(_obs.x.diff("x").mean()))
            L = decorrelation_length(obs_mean.values, pixel_size)
            block_size = max(1, int(np.ceil(L / pixel_size)))
            print(f"{ebm}/{v}: decorrelation length ≈ {L:.0f} m, block_size = {block_size} px")

            # 2) Block-bootstrap RMSE per exp_id (honors spatial correlation).
            rmse_boot = block_bootstrap_rmse(sim_mean_all, obs_mean, block_size, n_boot=500)
            rmse_mean = rmse_boot.mean(dim="boot")
            rmse_lo = rmse_boot.quantile(0.05, dim="boot")
            rmse_hi = rmse_boot.quantile(0.95, dim="boot")

            # 3) Rank by bootstrap-mean RMSE; treat exp_ids whose CI overlaps
            # the leader's upper bound as statistically tied with the best.
            best_id = rmse_mean.idxmin(dim="exp_id").values
            best_hi = float(rmse_hi.sel(exp_id=best_id))
            tied_mask = rmse_lo <= best_hi
            tied_ids = list(rmse_mean.exp_id.where(tied_mask, drop=True).values)
            print(f"{ebm}/{v}: best exp_id = {best_id}, n tied within 5-95% CI = {len(tied_ids)}")

            # Per-experiment weight for the parameter histograms: 1 if the
            # exp_id is in the statistically-tied set, 0 otherwise. This is
            # what ``np.repeat`` consumes below so each parameter value
            # contributes to the histogram only if its experiment passed the
            # bootstrap tie test.
            ebm_counts = pd.Series(
                tied_mask.values.astype(int),
                index=pd.Index(rmse_mean.exp_id.values, name="exp_id"),
            )

            fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))
            for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):
                # Repeat each parameter value by its sample count (= 1 if
                # the experiment tied with the best, 0 otherwise).
                values = np.repeat(
                    ebm_uq_df[key].values, ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int)
                )
                ax.hist(values, bins=15)
                ax.set_xlabel(value)
                ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                # ax.set_ylabel("Count")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}.png", dpi=300)
            plt.close()
            del fig

            # Write per-experiment stats to CSV so the user can inspect ties.
            rmse_df = (
                pd.DataFrame(
                    {
                        "rmse_mean": rmse_mean.values,
                        "rmse_lo": rmse_lo.values,
                        "rmse_hi": rmse_hi.values,
                        "tied_with_best": tied_mask.values,
                    },
                    index=pd.Index(rmse_mean.exp_id.values, name="exp_id"),
                )
                .join(ebm_uq_df, how="left")
                .sort_values("rmse_mean")
            )
            rmse_df.to_csv(f"{ebm}_{v}_rmse.csv")

            sim_best = _ds[v].sel(exp_id=best_id).mean(dim="time").squeeze()
            vmin = min(float(obs_mean.min()), float(sim_best.min()))
            vmax = max(float(obs_mean.max()), float(sim_best.max()))
            best_params = ebm_uq_df.loc[best_id]
            fig, axes = plt.subplots(1, 3, sharey=True, figsize=(12, 4))
            obs_mean.plot(ax=axes[0], vmin=vmin, vmax=vmax)
            axes[0].set_title("Observed")
            sim_best.plot(ax=axes[1], vmin=vmin, vmax=vmax)
            param_str = ", ".join(f"{v}={best_params[k]:.4g}" for k, v in ebm_uq_vars.items())
            rmse_best_mean = float(rmse_mean.sel(exp_id=best_id))
            rmse_best_lo = float(rmse_lo.sel(exp_id=best_id))
            rmse_best_hi = float(rmse_hi.sel(exp_id=best_id))
            axes[1].set_title(
                f"Best (id={best_id}, RMSE={rmse_best_mean:.1f} "
                f"[{rmse_best_lo:.1f}-{rmse_best_hi:.1f}], n_tied={len(tied_ids)})\n{param_str}"
            )
            (sim_best - obs_mean).plot(ax=axes[2], cmap="RdBu", vmin=-1000, vmax=1000)
            axes[2].set_title("Difference")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}_best_rmse.png", dpi=300)
            plt.close()
            del fig


In [ ]:
    ds = (
        xr.open_mfdataset(
            f"{data_dir}/2026_06_kitp_{ebm}_calib/output/processed_spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
            preprocess=partial(preprocess, uq_regexp=None, exp_regexp="uq_(.+?)_"),
            engine="netcdf4",
            join="outer",
            compat="no_conflicts",
            parallel=True,
            chunks="auto",
            decode_times=False,
            decode_timedelta=False,
        )
        .drop_dims("nv", errors="ignore")
        .pint.quantify()
    )
    ds["exp_id"] = ds["exp_id"].astype("int")


In [ ]:
    ds["exp_id"] = ds["exp_id"].astype("int")


In [ ]:
import xarray as xr
import matplotlib.pylab as plt
import xarray_regrid.methods.conservative  # pylint: disable=unused-import
from functools import partial
import xskillscore as xs
import pint_xarray
from dask.diagnostics import ProgressBar
import json
import pandas as pd
import numpy as np

from pism_terra.filtering import importance_sampling
from pism_terra.processing import preprocess_netcdf as preprocess

debm_uq_vars = {
    "surface.debm_simple.c1": "c1",
    "surface.debm_simple.c2": "c2",
    "surface.debm_simple.air_temp_all_precip_as_snow": "as_snow",
    "surface.debm_simple.air_temp_all_precip_as_rain": "as_rain",
    "surface.debm_simple.refreeze": "refreeze",
}


def decorrelation_length(field_2d, pixel_size, threshold=1.0 / np.e):
    """
    Radially-averaged spatial-ACF decorrelation length for a 2D field.

    Why: pixel-wise RMSE treats every cell as independent, but glaciological
    fields are smooth on scales of many cells. Returns the lag (in the units
    of ``pixel_size``) at which the autocorrelation first falls below
    ``threshold``; use that as the block side for bootstrap resampling.
    """
    a = np.asarray(field_2d, dtype=float)
    finite = np.isfinite(a)
    if not finite.any():
        return float("nan")
    a = np.where(finite, a, np.nanmean(a))
    a = a - a.mean()
    fft = np.fft.fft2(a)
    acf = np.fft.fftshift(np.fft.ifft2(fft * np.conj(fft)).real)
    acf = acf / acf.max()
    ny, nx = a.shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.indices(a.shape)
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2).astype(int)
    counts = np.maximum(np.bincount(r.ravel()), 1)
    radial = np.bincount(r.ravel(), weights=acf.ravel()) / counts
    rmax = min(cy, cx)
    radial = radial[: rmax + 1]
    below = np.where(radial < threshold)[0]
    lag_pixels = below[0] if below.size else rmax
    return float(lag_pixels) * float(pixel_size)


def block_bootstrap_rmse(sim, obs, block_size, n_boot=500, seed=0):
    """
    Block-bootstrap RMSE per experiment, returning an (exp_id, boot) array.

    ``sim`` is (exp_id, y, x), ``obs`` is (y, x). Blocks of side
    ``block_size`` pixels (≥ decorrelation length / pixel size) are resampled
    with replacement; one global RMSE is computed per resample per experiment.
    """
    sim_v = np.asarray(sim.values, dtype=float)
    obs_v = np.asarray(obs.values, dtype=float)
    sq_err = (sim_v - obs_v[None, :, :]) ** 2
    valid = np.isfinite(sq_err).all(axis=0)
    ny, nx = obs_v.shape
    by = max(1, ny // block_size)
    bx = max(1, nx // block_size)
    block_sums = np.zeros((sim_v.shape[0], by, bx))
    block_counts = np.zeros((by, bx), dtype=int)
    for i in range(by):
        for j in range(bx):
            ys = slice(i * block_size, (i + 1) * block_size)
            xs = slice(j * block_size, (j + 1) * block_size)
            v = valid[ys, xs]
            block_counts[i, j] = int(v.sum())
            chunk = np.where(v, sq_err[:, ys, xs], 0.0)
            block_sums[:, i, j] = chunk.sum(axis=(1, 2))
    block_sums = block_sums.reshape(sim_v.shape[0], -1)
    block_counts = block_counts.reshape(-1)
    valid_blocks = np.where(block_counts > 0)[0]
    rng = np.random.default_rng(seed)
    rmses = np.empty((sim_v.shape[0], n_boot))
    for b in range(n_boot):
        idx = rng.choice(valid_blocks, size=valid_blocks.size, replace=True)
        s = block_sums[:, idx].sum(axis=1)
        c = block_counts[idx].sum()
        rmses[:, b] = np.sqrt(s / max(c, 1))
    return xr.DataArray(
        rmses,
        dims=["exp_id", "boot"],
        coords={"exp_id": sim.exp_id, "boot": np.arange(n_boot)},
    )


data_dir = "~/base/pism-terra"

pctls = [0.05, 0.95]
fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

debm_uq_vars = {
    "surface.debm_simple.c1": "c1",
    "surface.debm_simple.c2": "c2",
    "surface.debm_simple.air_temp_all_precip_as_snow": "as_snow",
    "surface.debm_simple.air_temp_all_precip_as_rain": "as_rain",
    "surface.debm_simple.refreeze": "refreeze",
}

pdd_uq_vars = {"surface.pdd.factor_ice": "fice", "surface.pdd.factor_snow": "fsnow", "surface.pdd.refreeze": "refreeze"}

m_vars = ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]

obs = xr.open_dataset(
    f"{data_dir}/2026_06_kitp_debm_calib/kitp/input/v4/HIRHAM5-ERA5_YMM_1990_2019_v4.nc",
    engine="netcdf4",
    decode_times=False,
    decode_timedelta=False,
    chunks=None,
).drop_dims("nv", errors="ignore")

obs = obs.pint.quantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[v] = obs[v].pint.to("kg m^-2 yr^-1")
obs = obs.pint.dequantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[f"{v}_error"] = xr.where(obs[v] != 0, 0.10 * obs[v], 1e-8)

for ebm, ebm_uq_vars, fudge_factor in zip(["debm"], [debm_uq_vars], [1]):

    ds = (
        xr.open_mfdataset(
            f"{data_dir}/2026_06_kitp_{ebm}_calib/output/processed_spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
            preprocess=partial(preprocess, uq_regexp=None, exp_regexp="uq_(.+?)_"),
            engine="netcdf4",
            join="outer",
            compat="no_conflicts",
            parallel=True,
            chunks="auto",
            decode_times=False,
            decode_timedelta=False,
        )
        .drop_dims("nv", errors="ignore")
        .pint.quantify()
    )
    ds["exp_id"] = ds["exp_id"].astype("int")
    for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
        ds[v] = ds[v].pint.to("kg m^-2 yr^-1")
    ds = ds.pint.dequantify()

    ebm_uq_df = ds.pism_config.to_series().apply(json.loads).apply(pd.Series)[ebm_uq_vars.keys()]
    ds["time"] = obs["time"]

    _obs = obs.regrid.conservative(ds.drop_vars("pism_config")).squeeze()
    mask = ds[m_vars].isel(exp_id=0).notnull()
    _obs[m_vars] = _obs[m_vars].where(mask)
    melt_mask = _obs["climatic_mass_balance"].mean(dim="time") < 1e36
    _obs[m_vars] = _obs[m_vars].where(melt_mask)
    _ds = ds[m_vars].where(melt_mask)

    for v in ["climatic_mass_balance"]:

        with ProgressBar():
            ebm_filtered = importance_sampling(
                _ds,
                _obs,
                sim_var=v,
                obs_mean_var=v,
                obs_std_var=f"{v}_error",
                sum_dims=["time", "x", "y"],
                n_samples=ds.exp_id.size,
                fudge_factor=fudge_factor,
            )

            ebm_sampled_ids = ebm_filtered.exp_id_sampled.values
            print(ebm_filtered.exp_id_sampled.values)
            ebm_counts = pd.Series(ebm_sampled_ids).value_counts()
            print(ebm_counts)

            # Reindex config_df to the sampled IDs and plot histograms
            ds_sampled_configs = ebm_uq_df.loc[ebm_counts.index].reindex(ebm_counts.index)
            most_sampled_id = ebm_counts.idxmax()
            most_sampled_params = ebm_uq_df.loc[most_sampled_id]
            print(f"\n{ebm} / {v} — most sampled id={most_sampled_id} (count={ebm_counts.max()})")
            for k, short in ebm_uq_vars.items():
                print(f"  {short}: {most_sampled_params[k]:.6g}")

            fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))
            for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):
                # Repeat each parameter value by its sample count
                values = np.repeat(
                    ebm_uq_df[key].values, ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int)
                )
                ax.hist(values, bins=15)
                ax.set_xlabel(value)
                ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                # ax.set_ylabel("Count")

            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}.png", dpi=300)
            plt.close()
            del fig

        with ProgressBar():

            # 1) Decorrelation length from the observed time-mean field.
            sim_mean_all = _ds[v].mean(dim="time").compute()
            obs_mean = _obs[v].mean(dim="time").squeeze().compute()
            pixel_size = float(abs(_obs.x.diff("x").mean()))
            L = decorrelation_length(obs_mean.values, pixel_size)
            block_size = max(1, int(np.ceil(L / pixel_size)))
            print(f"{ebm}/{v}: decorrelation length ≈ {L:.0f} m, block_size = {block_size} px")

            # 2) Block-bootstrap RMSE per exp_id (honors spatial correlation).
            rmse_boot = block_bootstrap_rmse(sim_mean_all, obs_mean, block_size, n_boot=500)
            rmse_mean = rmse_boot.mean(dim="boot")
            rmse_lo = rmse_boot.quantile(0.05, dim="boot")
            rmse_hi = rmse_boot.quantile(0.95, dim="boot")

            # 3) Rank by bootstrap-mean RMSE; treat exp_ids whose CI overlaps
            # the leader's upper bound as statistically tied with the best.
            best_id = rmse_mean.idxmin(dim="exp_id").values
            best_hi = float(rmse_hi.sel(exp_id=best_id))
            tied_mask = rmse_lo <= best_hi
            tied_ids = list(rmse_mean.exp_id.where(tied_mask, drop=True).values)
            print(f"{ebm}/{v}: best exp_id = {best_id}, n tied within 5-95% CI = {len(tied_ids)}")

            # Per-experiment weight for the parameter histograms: 1 if the
            # exp_id is in the statistically-tied set, 0 otherwise. This is
            # what ``np.repeat`` consumes below so each parameter value
            # contributes to the histogram only if its experiment passed the
            # bootstrap tie test.
            ebm_counts = pd.Series(
                tied_mask.values.astype(int),
                index=pd.Index(rmse_mean.exp_id.values, name="exp_id"),
            )

            fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))
            for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):
                # Repeat each parameter value by its sample count (= 1 if
                # the experiment tied with the best, 0 otherwise).
                values = np.repeat(
                    ebm_uq_df[key].values, ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int)
                )
                ax.hist(values, bins=15)
                ax.set_xlabel(value)
                ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                # ax.set_ylabel("Count")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}.png", dpi=300)
            plt.close()
            del fig

            # Write per-experiment stats to CSV so the user can inspect ties.
            rmse_df = (
                pd.DataFrame(
                    {
                        "rmse_mean": rmse_mean.values,
                        "rmse_lo": rmse_lo.values,
                        "rmse_hi": rmse_hi.values,
                        "tied_with_best": tied_mask.values,
                    },
                    index=pd.Index(rmse_mean.exp_id.values, name="exp_id"),
                )
                .join(ebm_uq_df, how="left")
                .sort_values("rmse_mean")
            )
            rmse_df.to_csv(f"{ebm}_{v}_rmse.csv")

            sim_best = _ds[v].sel(exp_id=best_id).mean(dim="time").squeeze()
            vmin = min(float(obs_mean.min()), float(sim_best.min()))
            vmax = max(float(obs_mean.max()), float(sim_best.max()))
            best_params = ebm_uq_df.loc[best_id]
            fig, axes = plt.subplots(1, 3, sharey=True, figsize=(12, 4))
            obs_mean.plot(ax=axes[0], vmin=vmin, vmax=vmax)
            axes[0].set_title("Observed")
            sim_best.plot(ax=axes[1], vmin=vmin, vmax=vmax)
            param_str = ", ".join(f"{v}={best_params[k]:.4g}" for k, v in ebm_uq_vars.items())
            rmse_best_mean = float(rmse_mean.sel(exp_id=best_id))
            rmse_best_lo = float(rmse_lo.sel(exp_id=best_id))
            rmse_best_hi = float(rmse_hi.sel(exp_id=best_id))
            axes[1].set_title(
                f"Best (id={best_id}, RMSE={rmse_best_mean:.1f} "
                f"[{rmse_best_lo:.1f}-{rmse_best_hi:.1f}], n_tied={len(tied_ids)})\n{param_str}"
            )
            (sim_best - obs_mean).plot(ax=axes[2], cmap="RdBu", vmin=-1000, vmax=1000)
            axes[2].set_title("Difference")
            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}_best_rmse.png", dpi=300)
            plt.close()
            del fig


In [ ]:
    for v in ["climatic_mass_balance"]:

        with ProgressBar():
            ebm_filtered = importance_sampling(
                _ds,
                _obs,
                sim_var=v,
                obs_mean_var=v,
                obs_std_var=f"{v}_error",
                sum_dims=["time", "x", "y"],
                n_samples=ds.exp_id.size,
                fudge_factor=100000,
            )

            ebm_sampled_ids = ebm_filtered.exp_id_sampled.values
            print(ebm_filtered.exp_id_sampled.values)
            ebm_counts = pd.Series(ebm_sampled_ids).value_counts()
            print(ebm_counts)

            # Reindex config_df to the sampled IDs and plot histograms
            ds_sampled_configs = ebm_uq_df.loc[ebm_counts.index].reindex(ebm_counts.index)
            most_sampled_id = ebm_counts.idxmax()
            most_sampled_params = ebm_uq_df.loc[most_sampled_id]
            print(f"\n{ebm} / {v} — most sampled id={most_sampled_id} (count={ebm_counts.max()})")
            for k, short in ebm_uq_vars.items():
                print(f"  {short}: {most_sampled_params[k]:.6g}")

            fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))
            for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):
                # Repeat each parameter value by its sample count
                values = np.repeat(
                    ebm_uq_df[key].values, ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int)
                )
                ax.hist(values, bins=15)
                ax.set_xlabel(value)
                ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                # ax.set_ylabel("Count")

            fig.tight_layout()
            fig.savefig(f"{ebm}_{v}.png", dpi=300)
            plt.close()
            del fig


In [ ]:
        for fudge_factor in [1, 10, 100, 1000, 10_000, 100_000]:
            with ProgressBar():
                ebm_filtered = importance_sampling(
                    _ds,
                    _obs,
                    sim_var=v,
                    obs_mean_var=v,
                    obs_std_var=f"{v}_error",
                    sum_dims=["time", "x", "y"],
                    n_samples=ds.exp_id.size,
                    fudge_factor=fudge_factor,
                )

                ebm_sampled_ids = ebm_filtered.exp_id_sampled.values
                print(ebm_filtered.exp_id_sampled.values)
                ebm_counts = pd.Series(ebm_sampled_ids).value_counts()
                print(ebm_counts)

                # Reindex config_df to the sampled IDs and plot histograms
                ds_sampled_configs = ebm_uq_df.loc[ebm_counts.index].reindex(ebm_counts.index)
                most_sampled_id = ebm_counts.idxmax()
                most_sampled_params = ebm_uq_df.loc[most_sampled_id]
                print(f"\n{ebm} / {v} — most sampled id={most_sampled_id} (count={ebm_counts.max()})")
                for k, short in ebm_uq_vars.items():
                    print(f"  {short}: {most_sampled_params[k]:.6g}")

                fig, axes = plt.subplots(1, len(ebm_uq_vars), sharey=True, figsize=(6.4, 1.8))
                for ax, (key, value) in zip(axes.flat, ebm_uq_vars.items()):
                    # Repeat each parameter value by its sample count
                    values = np.repeat(
                        ebm_uq_df[key].values, ebm_counts.reindex(ebm_uq_df.index, fill_value=0).values.astype(int)
                    )
                    ax.hist(values, bins=15)
                    ax.set_xlabel(value)
                    ax.set_xlim(ebm_uq_df[key].min(), ebm_uq_df[key].max())
                    # ax.set_ylabel("Count")

                fig.tight_layout()
                fig.savefig(f"{ebm}_{v}_ff_{fudge_factor}.png", dpi=300)
                plt.close()
                del fig


In [ ]:
_ds.time

In [ ]:
cmb=[  35277350.6668686, 34865842.8523401, 27164576.3935833, 31384020.21825, 
    27091591.7694377, -695137.209980904, -14811729.5007511, 20771336.5970434, 
    37689346.0225213, 34195358.1789098, 38290756.0771725, 34156582.7616074, 
    35272515.0371053, 34864931.6116, 27162939.1792409, 31358871.9844531, 
    26690506.9565616, 130632.974422076, -14365201.0055751, 20966279.2594184, 
    37701101.9241263, 34189883.2921089, 38288830.9755408, 34154360.5940036]

In [ ]:
 tendency_of_ice_mass = np.array([
  -393.647587160264, 42.373846369612, 76.5196142265892, 71.7193698708365, 
    64.8048461280541, -40.2462763055583, -92.2372392255767, 
    -37.6235447949085, 113.076424197926, 103.123688649644, 132.13511585347, 
    132.66951891688, 154.844848504677, 160.051661838876, 99.5091281382907, 
    106.286111270717, 75.2115035003671, -26.9059959792702, -78.8109168956178, 
    -26.3299168119409, 122.796195825525, 109.181318583834, 137.890850509479, 
    137.194062376535,
  -814.76449186546, -34.0235906612722, -32.3269073483592, -38.3240005893035, 
    -27.378680428647, -75.5646480960982, -145.205100231968, 
    -48.5058340908892, 37.5090434435162, 38.2606855168542, 44.1512928004977, 
    18.0339136683085, 18.7200528272473, 5.28131633760352, 13.4076286638569, 
    15.3283112735353, 43.1447528983195, -20.3131572951904, -104.072415153478, 
    -18.8553846125578, 64.3226802838722, 35.8807023051394, 62.5711088844693, 
    33.4372111091789,
  -737.814922950031, -33.1122686185947, -35.4366403148824, -48.6311617255607, 
    -36.1266141598668, -82.272556319807, -238.141926587644, 
    -107.318841885821, -43.078726030808, -33.5317095883345, 
    -28.8495448465279, -38.9272756311866, -33.4751493600749, 
    -60.7911403231832, -65.9200452688318, -49.4293441930997, 
    -48.0118528721477, -80.1265583759408, -228.649758540182, 
    -215.900503121799, -55.2183433831065, -77.8118146611595, 
    -31.1915522180972, -57.7241194602812,
  -924.942254155543, -15.6927804339297, -14.205624796626, -78.9245173406843, 
    -1.37865283045296, 10.1769967291775, -59.3208204301622, -11.424357380054, 
    2.13248483646111, -7.00951542815323, -3.98625489019826, 
    -12.5499984979043, -13.685866829416, -16.8704119003744, 
    -10.5132914813064, -4.41012155020537, -16.7076883952977, 
    22.7210985878963, -48.2122594747976, 20.4709742694565, 9.26259672087881, 
    4.59770740438996, -1.79282138655759, -8.1637504894704,
  -2370.84527202415, -111.483436125111, -188.696828821442, -237.111864700971, 
    -275.599072741795, -206.980411282267, -276.123940330741, 
    -147.960709019668, -140.546534850896, -124.452749430452, 
    -166.850141023301, -131.757988266088, -129.220843635818, 
    -150.893931634909, -141.023845232135, -107.936472353085, 
    -76.427098549833, -96.3516473236979, -173.111220399877, 
    -70.7956800050419, -80.5818231750235, -46.7266382157452, 
    -70.8477962973388, -92.9425463455011,
  -393.157318577409, 69.1905014062994, 102.268013748005, 127.022648738932, 
    40.08975953113, -55.2502830558154, -104.31535318989, -64.0402670810195, 
    92.3347016228034, 114.427603372344, 200.04161010284, 219.589479741224, 
    251.100477642712, 258.16654234819, 149.080549989306, 174.901567343453, 
    93.3485373547482, 2.87843350720329, -79.9342285074149, -43.9322771210041, 
    109.937894580301, 129.152357470781, 213.410249883449, 230.777607026089,
  28.3373034938545, 86.3832958945003, 83.1628675665479, 103.437320024093, 
    90.2940349534541, -5.6732442088866, -169.622346833336, -65.9354699857284, 
    138.438409934899, 124.502428184073, 144.321656909126, 98.9620254808003, 
    88.5834407368887, 88.6857185646624, 83.3939937866622, 103.21715512686, 
    89.8546847066673, -3.93207103144227, -167.711632438573, 
    -65.3915555306964, 137.650297923353, 123.653409192353, 143.45768941677, 
    98.1782121267115,
  -5606.83454323901, 3.63556783150446, -8.71550574016752, -100.812205722658, 
    -145.294379548124, -455.810422539255, -1084.96672682932, 
    -482.809024238088, 199.865803153903, 215.320431275976, 320.963734905906, 
    286.019675412034, 336.866959886216, 283.629755230866, 127.934118595842, 
    237.957206918175, 160.412838642824, -202.029897910442, -880.502431409941, 
    -420.734342933584, 308.169498775801, 277.927042079592, 453.497728792173, 
    340.756676343262])

In [ ]:
plt.plot(tendency_of_ice_mass.reshape(8, 24)[7,12:])

In [ ]:
np.mean(tendency_of_ice_mass.reshape(8, 24)[7,12:])

In [ ]:
 tendency_of_ice_mass = np.array([-10609.5541876112, -325.474583270271, 
    47.2794876253727, -0.420246104642424, -131.255819901986, 
    -866.483933554425, -1826.59926765713, -739.238970035216, 
    458.284812702703, 541.497954823011, 713.492525092578, 632.577162912414, 
    688.474242334976, 585.867297671568, 363.51169216657, 476.912620538141, 
    288.631457657773, -518.620482588878, -1564.29176532278, 
    -635.424684068281, 589.431209030278, 648.953924265763, 863.7744635013, 
    727.287399894242])

In [ ]:
np.mean(tendency_of_ice_mass.reshape(8, 24)[7,12:])

In [ ]:
np.mean(tendency_of_ice_mass[12:])

In [ ]:
 tendency_of_ice_mass = np.array([
  -393.647587160264, 42.373846369612, 76.5196142265892, 71.6426611002404, 
    49.7980232914279, -103.584774591925, -171.694907777393, 
    -86.7678416516304, 103.208536080737, 103.191204805075, 133.730717585092, 
    131.797046357139, 153.177559281595, 170.26432809105, 99.8869681303386, 
    105.307302649865, 59.8300044522589, -91.4585659474925, -157.875067530366, 
    -76.1089227486313, 112.717129638159, 109.359715969732, 138.7096399961, 
    137.962315134945,
  -814.76449186546, -34.0235906612722, -32.3269073483592, -38.4083692238679, 
    -34.0306296014768, -103.881598057936, -177.034540167741, 
    -63.9170205801085, 33.0665533699133, 37.458183895077, 43.9360143960847, 
    18.0440205460067, 18.7375423225637, 5.29806916863027, -9.3174468571979, 
    41.3960496576584, 36.5284058662685, -49.4572356975457, -135.394018867485, 
    -34.0647828881098, 60.7183110302081, 36.1274054960423, 56.3369014034133, 
    33.2424671460573,
  -737.814922950031, -33.1122686185947, -35.4366403148824, -48.6311617255607, 
    -39.7874558935784, -120.458509477167, -277.593216300719, 
    -134.152056387015, -45.0513846918306, -33.4869466519615, 
    -28.8127129867685, -38.8849668105811, -33.4640610369255, 
    -60.8928120533528, -65.8567080515619, -49.398242007489, 
    -51.6748315387663, -117.586301417242, -290.587075626701, 
    -216.801380889423, -56.9940545687648, -77.563475860347, 
    -31.1503745650723, -57.6221610755747,
  -924.942254155543, -15.6927804339297, -14.205624796626, -78.9245173406843, 
    -1.65173427688031, 2.1010811855321, -70.5604599965713, -18.8676269724358, 
    2.11339520977846, -6.99872027506405, -4.00909282590098, 
    -12.5346315074122, -13.712124746317, -16.8544301858098, 
    -10.5396047326291, -4.39317678360612, -16.9756770943107, 
    14.8627199449416, -59.3817556269848, 13.1156866090691, 9.2671270342344, 
    4.60610447004605, -0.330790416307515, -8.22513951480479,
  -2370.84527202415, -111.483436125111, -188.696828821442, -237.111864700971, 
    -277.766160819871, -225.76840895326, -302.410244188762, 
    -162.780386322862, -149.980393681492, -194.729383579869, 
    -186.996888642282, -130.735067596495, -128.248081901284, 
    -147.066373441121, -140.069764584634, -107.004694233747, 
    -77.2795690712637, -108.087040485036, -198.373849417579, 
    -84.5289622481111, -88.07133512793, -46.1730011573856, -69.7759039005809, 
    -93.1556242262882,
  -393.157318577409, 69.1905014062994, 102.26257252034, 125.224182179464, 
    -1.57136715527144, -87.0576098504185, -177.835625707824, 
    -99.7578241264361, 81.7854925268119, 113.790577720536, 200.271787622528, 
    219.634622738583, 251.278150462216, 258.38464582085, 149.236815885818, 
    173.341885319404, 75.3843351597122, -53.1650778556582, -153.640686147155, 
    -79.2081429875495, 99.6095602108333, 128.546847589478, 213.692183321113, 
    231.015247812739,
  28.3373034938545, 86.3832958945003, 83.1628675665479, 102.462516376599, 
    66.5471116277919, -106.330663836148, -308.55592216663, -138.558199639986, 
    123.48064805629, 123.931813479756, 144.31166944413, 98.9234036993508, 
    88.5523856182327, 88.6940265873471, 83.3861701607713, 102.296024117862, 
    66.229327304085, -103.780069088472, -306.525240373709, -138.3840294835, 
    122.552931440065, 122.992602840658, 143.453714145963, 98.1563625798442,
  -5606.83454323901, 3.63556783150446, -8.72094696783279, -103.746553334781, 
    -238.462212827858, -744.980483581322, -1485.68491630564, 
    -704.800955680474, 148.622846870208, 143.156729393549, 302.431494592883, 
    286.244427426591, 336.321370000081, 297.827453987594, 106.726429950905, 
    261.545148719948, 92.0419950779838, -508.671570546505, -1301.77769358998, 
    -615.980534636256, 259.799669656805, 277.896199348224, 450.935369984629, 
    341.373467856918 ])

In [ ]:
np.mean(tendency_of_ice_mass.reshape(8, 24)[7,12:])

In [ ]:
 tendency_of_ice_mass = np.array([
  -393.647587160264, 42.373846369612, 76.5196142265892, 71.7219745034279, 
    66.989560117869, -25.402891937058, -72.7775107868047, -30.5095695141672, 
    113.944432112229, 101.983166683605, 133.509186399762, 131.433401950661, 
    156.094006951835, 169.772364242234, 102.453834010003, 95.8117687106735, 
    93.2380566275785, -11.7090544913098, -61.8572836126184, 
    -21.7116533078431, 123.419283959608, 108.461023112279, 137.87619097421, 
    137.243514759835,
  -814.76449186546, -34.0235906612722, -32.3269073483592, -38.3190545869279, 
    -26.6223215161339, -67.0096990616092, -131.61744473093, 
    -43.2102882588637, 37.7830021438578, 38.1771107394893, 44.084052935778, 
    17.9794375577603, 18.6689482742621, 5.23676376939706, 13.3599392786391, 
    15.2301437688002, 43.8747800919942, -12.2731430984734, -90.3453621209619, 
    -13.2669092611646, 64.6356206134661, 35.8349627974095, 62.4675344044463, 
    33.3396901452306,
  -737.814922950031, -33.1122686185947, -35.4366403148824, -48.6311617255606, 
    -35.6998974667278, -72.680063991413, -217.161947519923, 
    -104.056917741386, -43.051009862398, -33.5402240606093, 
    -28.8826507730843, -38.9529523397216, -33.5266670207958, 
    -60.9558528358566, -65.9218080510955, -49.4200084365825, 
    -47.602979573808, -70.5657420579064, -207.894902720403, 
    -213.006662350558, -55.2659425185421, -77.9735518497167, 
    -31.2062495355089, -57.7887947067826,
  -924.942254155543, -15.6927804339297, -14.205624796626, -78.9245173406843, 
    -1.36931606242616, 14.564402912766, -42.2589330524763, -9.06823190131618, 
    2.10149296839632, -7.01892584564977, -4.00188105048198, 
    -12.5653262330595, -13.7605366741757, -16.8049487534176, 
    -10.5724358230184, -4.54955607482875, -16.8245308271243, 
    26.7508943801734, -31.3797449942182, 22.881321091743, 9.24203823307713, 
    4.57244087215773, -1.79996686397137, -8.41635893865581,
  -2370.84527202415, -111.483436125111, -188.696828821442, -237.111864700971, 
    -275.444998657739, -199.901440284763, -261.88237163428, 
    -145.139017588778, -143.269316666464, -128.972886886883, 
    -144.885315479858, -137.581300167003, -131.647634779341, 
    -153.444268343992, -143.681690296124, -110.552307335218, 
    -78.9302939541254, -92.9236415845049, -156.463122975732, 
    -91.2881246326132, -80.4435924888814, -47.6477317977512, 
    -71.079075830436, -91.471029491509,
  -393.157318577409, 69.1905014062994, 102.268200091984, 127.180470273204, 
    43.3113380571555, -43.8093900701108, -85.9795243200619, 
    -53.8981416078264, 94.6739592099604, 114.696875850748, 200.207653842027, 
    219.692826205335, 251.165999087106, 258.250920977341, 149.149499372844, 
    175.100806202002, 96.6319523929016, 14.2106063210414, -63.3466911582459, 
    -33.3149366812349, 112.173070720619, 129.41168637288, 213.550420396014, 
    230.898451279324,
  28.3373034938545, 86.3832958945003, 83.1628675665479, 103.55980635823, 
    93.0964053086301, 12.5443360507781, -140.369436548837, -53.3133694079972, 
    140.121211657703, 124.55385799158, 144.321483257023, 98.9547809116395, 
    88.5886367774491, 88.6790889510267, 83.403918320176, 103.325861414168, 
    92.6739897967338, 13.899858049789, -138.57905706417, -52.6851720792153, 
    139.389479240627, 123.710678179401, 143.497296935621, 98.1698724933608,
  -5606.83454323901, 3.63556783150446, -8.71531939618851, -100.524347219283, 
    -135.739230219373, -381.69474638141, -952.047168593314, 
    -439.195536020334, 202.303771563285, 209.87897447228, 344.352529131167, 
    278.960867885611, 335.58275261634, 290.734068006732, 128.191256811424, 
    224.946708249015, 183.06097455415, -132.610222481191, -749.86616464635, 
    -402.392137220886, 313.149957759974, 276.369507686659, 453.306150480375, 
    341.975345540803])

In [ ]:
np.mean(tendency_of_ice_mass.reshape(8, 24)[7,12:])

In [ ]:
fld_rmse = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_v4/output/processed_scalar/fldsum_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_rmse_0001-01-01_0003-01-01.nc").sel(basin="GIS").sel(time="0002")
fld_si = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_v4/output/processed_scalar/fldsum_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_si_0001-01-01_0003-01-01.nc").sel(basin="GIS").sel(time="0002")
fld_median = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_v4/output/processed_scalar/fldsum_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_median_0001-01-01_0003-01-01.nc").sel(basin="GIS").sel(time="0002")

In [ ]:
ds_rmse = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_v4/output/scalar/scalar_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_rmse_0001-01-01_0003-01-01.nc").sel(time="0002")
ds_si = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_v4/output/scalar/scalar_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_si_0001-01-01_0003-01-01.nc").sel(time="0002")
ds_median = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_v4/output/scalar/scalar_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_median_0001-01-01_0003-01-01.nc").sel(time="0002")

In [ ]:
fig, ax = plt.subplots(1,1)
fld_rmse.tendency_of_ice_mass_due_to_surface_mass_flux.plot(ax=ax)
fld_si.tendency_of_ice_mass_due_to_surface_mass_flux.plot(ax=ax)
fld_median.tendency_of_ice_mass_due_to_surface_mass_flux.plot(ax=ax)
ds_rmse.tendency_of_ice_mass_due_to_surface_mass_flux.plot(ax=ax, ls="dotted")
ds_si.tendency_of_ice_mass_due_to_surface_mass_flux.plot(ax=ax,ls="dotted")
ds_median.tendency_of_ice_mass_due_to_surface_mass_flux.plot(ax=ax,ls="dotted")

In [ ]:
fig, ax = plt.subplots(1,1)
fld_rmse.tendency_of_ice_mass_due_to_discharge.plot(ax=ax)
fld_si.tendency_of_ice_mass_due_to_discharge.plot(ax=ax)
fld_median.tendency_of_ice_mass_due_to_discharge.plot(ax=ax)
ds_rmse.tendency_of_ice_mass_due_to_discharge.plot(ax=ax, ls="dotted")
ds_si.tendency_of_ice_mass_due_to_discharge.plot(ax=ax,ls="dotted")
ds_median.tendency_of_ice_mass_due_to_discharge.plot(ax=ax,ls="dotted")

In [ ]:
fig, ax = plt.subplots(1,1)
fld_rmse.tendency_of_ice_mass_due_to_discharge.plot(ax=ax)
fld_si.tendency_of_ice_mass_due_to_discharge.plot(ax=ax)
fld_median.tendency_of_ice_mass_due_to_discharge.plot(ax=ax)
ds_rmse.tendency_of_ice_mass_due_to_discharge.plot(ax=ax, ls="dotted")
ds_si.tendency_of_ice_mass_due_to_discharge.plot(ax=ax,ls="dotted")
ds_median.tendency_of_ice_mass_due_to_discharge.plot(ax=ax,ls="dotted")

In [ ]:
import xarray as xr
import matplotlib.pylab as plt
fld_rmse = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_df/output/processed_scalar/fldsum_spatial_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc").sel(basin="GIS")


In [ ]:
fld_rmse.time

In [ ]:
import xarray as xr
import matplotlib.pylab as plt
fld_rmse = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_df/output/processed_scalar/fldsum_spatial_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc").sel(basin="GIS")
fld_median = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_median/output/processed_scalar/fldsum_spatial_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc").sel(basin="GIS")

ds_rmse = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_df/output/scalar/scalar_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc").sel(basin="GIS")
ds_median = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_median/output/scalar/calar_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc").sel(basin="GIS")


In [ ]:
import xarray as xr
import matplotlib.pylab as plt
fld_rmse = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_df/output/processed_scalar/fldsum_spatial_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc").sel(basin="GIS")
fld_median = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_median/output/processed_scalar/fldsum_spatial_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc").sel(basin="GIS")

ds_rmse = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_df/output/scalar/scalar_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc")
ds_median = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_median/output/scalar/calar_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc")

In [ ]:
import xarray as xr
import matplotlib.pylab as plt
fld_rmse = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_df/output/processed_scalar/fldsum_spatial_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc").sel(basin="GIS")
fld_median = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_median/output/processed_scalar/fldsum_spatial_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc").sel(basin="GIS")

ds_rmse = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_df/output/scalar/scalar_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc")
ds_median = xr.open_dataset("~/base/pism-terra/2026_06_kitp_debm_median/output/scalar/scalar_g900m_id_HIRHAM5-ERA5_YMM_1990_2019_0001-01-01_0002-01-01.nc")

fig, ax = plt.subplots(1,1)
fld_rmse.tendency_of_ice_mass.plot(ax=ax)
fld_median.tendency_of_ice_mass.plot(ax=ax)
ds_rmse.tendency_of_ice_mass.plot(ax=ax, ls="dotted")
ds_median.tendency_of_ice_mass.plot(ax=ax,ls="dotted")

fig, ax = plt.subplots(1,1)
fld_rmse.tendency_of_ice_mass_due_to_surface_mass_flux.plot(ax=ax)
fld_median.tendency_of_ice_mass_due_to_surface_mass_flux.plot(ax=ax)
ds_rmse.tendency_of_ice_mass_due_to_surface_mass_flux.plot(ax=ax, ls="dotted")
ds_median.tendency_of_ice_mass_due_to_surface_mass_flux.plot(ax=ax,ls="dotted")

fig, ax = plt.subplots(1,1)
fld_rmse.tendency_of_ice_mass_due_to_discharge.plot(ax=ax)
fld_median.tendency_of_ice_mass_due_to_discharge.plot(ax=ax)
ds_rmse.tendency_of_ice_mass_due_to_discharge.plot(ax=ax, ls="dotted")
ds_median.tendency_of_ice_mass_due_to_discharge.plot(ax=ax,ls="dotted")

print(fld_rmse.tendency_of_ice_mass.mean())
print(fld_median.tendency_of_ice_mass.mean())